# 22 · Sweep: окно подтверждения «серьёзной тревоги» 1..23 часа

Проверяем все промежутки 1–23 ч для эпизодов подтверждённого задымления
(≥2 дымовых каналов объекта в пределах окна) + ручной ИПР. Для каждого окна —
пересборка события, субъекты TTE, обучение fire (GPU) и метрики на том же
holdout. Ищем окно-оптимум и сверяемся с 6ч (принятое в §9/§17).

Протокол sweep-обучения: iters=900, lr=0.03, depth=10, ES=250 (усечённо для
скорости); метрики всех окон сопоставимы между собой; финальное 6ч из §9
(CPU, ES=300) = PR-AUC 0.421, Uno-C 0.792.


In [1]:
%matplotlib inline
import sys
import pathlib

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool

import data_utils as du
import features as fe
import tte_pipeline as tte
from tte_experiments import survival_metrics, pr_at_horizon


DATASET = du.find_dataset_dir()
hour_ns = fe.BUCKET_HOURS * 3600 * 10 ** 9
EV = pd.read_parquet(DATASET / "fire_raw_events_30m.parquet")
panel_base = pd.read_csv(DATASET / "subdaily_panel_fire6h.csv",
                         dtype={"ид_канала_данных": str, "ид_объект": str})
panel_base = tte.refit_z_train_only(panel_base)
print("сырых событий:", len(EV), "| панель:", panel_base.shape)


сырых событий: 164477 | панель: (1204508, 143)


In [2]:
def episodes(gap_ns):
    out = []
    for об, grp in EV.groupby("об"):
        g = grp.sort_values("t")
        t = g["t"].to_numpy()
        gid = np.cumsum(np.concatenate([[True], t[1:] - t[:-1] > gap_ns])) - 1
        g2 = g.copy(); g2["gid"] = gid
        cnt = g2[g2["тип"] == "дым"].groupby("gid")["ид"].nunique().rename("n_dym")
        agg = g2.groupby("gid").agg(
            t0=("t", "min"), n_manual=("тип", lambda s: int((s == "ручной").sum())))
        agg = agg.join(cnt).fillna({"n_dym": 0})
        agg = agg[(agg["n_dym"] >= 2) | (agg["n_manual"] > 0)]
        d = agg.reset_index()
        d["об"] = об
        out.append(d)
    ep = pd.concat(out, ignore_index=True) if out else pd.DataFrame()
    return ep

def run_model(X_cols, trs_df, hos_df, tr_pt, va_pt):
    m = CatBoostClassifier(iterations=900, learning_rate=0.03, depth=10,
                           random_seed=42, task_type="GPU",
                           loss_function="Logloss", eval_metric="Logloss",
                           early_stopping_rounds=250, verbose=0,
                           allow_writing_files=False)
    m.fit(Pool(*tr_pt), eval_set=Pool(*va_pt))
    Xh, _ = tte.expand_person_time(hos_df, X_cols, tte.HORIZON_BUCKETS,
                                   fixed_horizon=True)
    S = np.cumprod(1 - m.predict_proba(Xh)[:, 1].reshape(-1, tte.HORIZON_BUCKETS),
                   axis=1)
    surv_tr = tte.make_surv_struct(trs_df["event_flag"], trs_df["obs_days"])
    surv_ho = tte.make_surv_struct(hos_df["event_flag"], hos_df["obs_days"])
    met = survival_metrics(surv_tr, surv_ho, 1 - S[:, -1], S, tte.EVAL_TIMES)
    y24 = ((hos_df["event_flag"] == 1) & (hos_df["obs_days"] <= 1.01)).to_numpy(int)
    p24 = 1 - S[:, int(round(1.0 * tte.STEPS_PER_DAY)) - 1]
    pr = pr_at_horizon(y24, p24)
    return met, pr


In [3]:
# Главный цикл: окна 1..23 часа
rows = []
for h in range(1, 24):
    ep = episodes(h * 3600 * 10 ** 9)
    if len(ep) == 0:
        print(f"h={h}: нет эпизодов"); continue
    start_map = (ep[["об", "бакет"]] if "бакет" in ep else
                 ep.assign(бакет=(ep["t0"] // hour_ns).astype(np.int64))[["об", "бакет"]]
                 ).drop_duplicates().rename(columns={"об": "ид_объект"})
    pn = panel_base.copy()
    pn = pn.merge(start_map, on=["ид_объект", "бакет"], how="left", indicator="m")
    pn["серьёзных_окно"] = (pn["m"] == "both").astype(np.int8)
    pn = pn.drop(columns=["m"])
    pn = tte.add_series_context(pn, event_col="серьёзных_окно")
    subj = tte.add_tte_labels(pn)
    subj["ид_объект_code"] = subj["ид_объект"].astype("category").cat.codes
    subj["тип_датчика_code"] = subj["тип_датчика"].astype("category").cat.codes
    tr, va, ho = tte.split_subjects(subj)
    X_cols = tte.subject_features(tr)
    trs = tte.sample_subjects(tr, 12000, 8000, 8000, seed=42)
    vas = tte.sample_subjects(va, 2000, 2000, 2000, seed=43)
    hos = tte.sample_holdout(ho, 20000, seed=777)
    X_tr, y_tr = tte.expand_person_time(trs, X_cols, tte.HORIZON_BUCKETS)
    X_va, y_va = tte.expand_person_time(vas, X_cols, tte.HORIZON_BUCKETS)
    met, pr = run_model(X_cols, trs, hos, (X_tr, y_tr), (X_va, y_va))
    ev = trs["event_flag"].fillna(0).to_numpy(); od = trs["obs_days"].to_numpy()
    base24tr = ((ev == 1) & (od <= 1.01)).mean() * 100
    rows.append({"h": h, "эпизодов": len(ep),
                 "база24_train%": round(base24tr, 3),
                 "база24_holdout%": round(pr["base"] * 100, 3),
                 "uno_c": round(met["uno_c"], 4), "pr224": round(pr["pr_auc"], 4)})
    print(f"h={h:2d}: эп {len(ep):5d} | PR24={pr['pr_auc']:.3f} Uno-C={met['uno_c']:.3f} "
          f"база24_ho={pr['base']*100:.2f}%", flush=True)

RES = pd.DataFrame(rows)
RES.to_csv(DATASET / "fire_window_sweep.csv", index=False, encoding="utf-8-sig")
print()
print(RES.round(4).to_string(index=False))


h= 1: эп  2072 | PR24=0.109 Uno-C=0.684 база24_ho=3.05%


h= 2: эп  1953 | PR24=0.059 Uno-C=0.635 база24_ho=2.67%


h= 3: эп  1913 | PR24=0.149 Uno-C=0.655 база24_ho=2.86%


h= 4: эп  1899 | PR24=0.085 Uno-C=0.675 база24_ho=3.14%


h= 5: эп  1887 | PR24=0.158 Uno-C=0.654 база24_ho=3.10%


...

h=20: эп  1790 | PR24=0.210 Uno-C=0.624 база24_ho=2.28%


h=21: эп  1727 | PR24=0.132 Uno-C=0.584 база24_ho=2.27%


h=22: эп  1681 | PR24=0.223 Uno-C=0.602 база24_ho=2.06%


h=23: эп  1643 | PR24=0.126 Uno-C=0.637 база24_ho=2.07%



 h  эпизодов  база24_train%  база24_holdout%  uno_c  pr224
 1      2072          6.411            3.050 0.6839 0.1086
 2      1953          7.050            2.670 0.6346 0.0592
 3      1913          7.157            2.855 0.6553 0.1494
 4      1899          6.871            3.140 0.6752 0.0850
 5      1887          6.864            3.095 0.6540 0.1577
 6      1892          6.821            3.115 0.6490 0.0991
 7      1893          7.018            3.115 0.6664 0.0860
 8      1887          6.989            2.810 0.6817 0.0983
 9      1888          6.914            2.815 0.6612 0.0855
10      1887          7.004            2.820 0.6510 0.0519
11      1885          6.986            2.900 0.6749 0.0561
12      1885          6.854            2.895 0.6433 0.0738
13      1882          6.761            2.900 0.6702 0.0561
14      1883          6.529            2.900 0.6330 0.0941
15      1874          6.650            2.890 0.6408 0.0969
16      1872          6.704            3.090 0.6579 0.1